<a href="https://colab.research.google.com/github/fnlopezsanz/TP_Lab_Met_Cuantitativos_Inversiones/blob/main/IPC_V1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


# 1. Leer el archivo CSV especificando el separador ';' y los decimales ',' además el encoding
df_ipc = pd.read_csv("https://github.com/fnlopezsanz/TP_Lab_Met_Cuantitativos_Inversiones/raw/main/IPC_NACIONAL_INDEC.csv", delimiter = ";", decimal = ",", encoding='latin-1')

# Limpiar posibles espacios en blanco en los nombres de las columnas
df_ipc.columns = df_ipc.columns.str.strip()


# 2. Convertir la columna de fecha a formato datetime y ordenar la tabla
df_ipc['Periodo'] = pd.to_datetime(df_ipc['Periodo'], format='%m/%Y')
df_ipc = df_ipc.sort_values('Periodo').reset_index(drop=True)

#Evaluamos el df del IPC
print("Forma de df_ipc", df_ipc.shape)
print("\nInformación de df_ipc:")
print(df_ipc.info())
print()
#Se puede observar que este df no tiene valores nulos, ya que dentro de cada columna hay 118 valores no nulos
#siendo la misma cantidad que las filas presentes en el df_ipc.
#Además, no hay tipos de datos que requieran ser cambiados, luego de haber transformado las fechas a formato datetime


# 3. Calcular la Inflación Mensual (%)
df_ipc['Inflacion_Mensual_%'] = df_ipc['Indice IPC'].pct_change() * 100


# 4. Calcular la Inflación Acumulada del año 2025 (%)
# Tomamos como base el valor del índice de Diciembre de 2024 y lo guardo en una variable
base_dic_2024 = df_ipc.loc[df_ipc['Periodo'] == '2024-12-01', 'Indice IPC'].values[0]

# Filtramos los meses correspondientes a 2025 y aplicamos la fórmula de variación acumulada
df_ipc_2025 = df_ipc['Periodo'].dt.year == 2025
df_ipc.loc[df_ipc_2025, 'Inflacion_Acumulada_2025_%'] = ((df_ipc.loc[df_ipc_2025, 'Indice IPC'] / base_dic_2024) - 1) * 100


# 5. Formatear la fecha a mes/año para visualizar más claramente
df_ipc['Periodo'] = df_ipc['Periodo'].dt.strftime('%m/%Y')


# 6. Filtramos el año 2025 y reseteamos el índice de ESA tabla resultante
df_ipc_2025 = df_ipc[df_ipc['Periodo'].str.endswith('/2025')].reset_index(drop=True)


# 7. Calculo de la actualizacion de capital

# Definimos la variable capital
Capital_Inicial = 1000000

# Calculamos la actualización y el monto actualizado del capital

df_ipc_2025['Actualizacion'] = (df_ipc_2025['Inflacion_Acumulada_2025_%'] /100 ) * Capital_Inicial
df_ipc_2025['Capital_Actualizado'] = (df_ipc_2025['Inflacion_Acumulada_2025_%'] /100 ) * Capital_Inicial + Capital_Inicial

# 8. Aplicamos formato y visualizamos los resultados

# Aplicamos el estilo de visualización
df_ipc_2025_estilizado = df_ipc_2025.style.format({
    'Indice IPC': '{:.2f}',
    'Inflacion_Mensual_%': '{:.2f}%',
    'Inflacion_Acumulada_2025_%': '{:.2f}%',
    'Actualizacion': '{:,.2f}',
    'Capital_Actualizado': '{:,.2f}'
})

# Se usa display() directamente sobre el objeto Styler, sin usar .head()
display(df_ipc_2025_estilizado)

Forma de df_ipc (118, 2)

Información de df_ipc:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 118 entries, 0 to 117
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   Periodo     118 non-null    datetime64[ns]
 1   Indice IPC  118 non-null    float64       
dtypes: datetime64[ns](1), float64(1)
memory usage: 2.0 KB
None



,Periodo,Indice IPC,Inflacion_Mensual_%,Inflacion_Acumulada_2025_%,Actualizacion,Capital_Actualizado
0,01/2025,7864.10,2.21%,2.21%,"22,108.14","1,022,108.14"
1,02/2025,8053.00,2.40%,4.67%,"46,659.73","1,046,659.73"
2,03/2025,8353.30,3.73%,8.57%,"85,690.15","1,085,690.15"
3,04/2025,8585.60,2.78%,11.59%,"115,882.51","1,115,882.51"
4,05/2025,8714.50,1.50%,13.26%,"132,635.82","1,132,635.82"
5,06/2025,8855.60,1.62%,15.10%,"150,974.79","1,150,974.79"
6,07/2025,9024.00,1.90%,17.29%,"172,861.97","1,172,861.97"
7,08/2025,9193.20,1.88%,19.49%,"194,853.13","1,194,853.13"
8,09/2025,9384.10,2.08%,21.97%,"219,664.67","1,219,664.67"
9,10/2025,9603.90,2.34%,24.82%,"248,232.39","1,248,232.39"


In [9]:
# 9. Calcular la suma de la columna actualizacion y el monto actualizado a diciembre 2025
Monto_Actualizado = df_ipc_2025.loc[df_ipc_2025['Periodo'] == '12/2025', 'Capital_Actualizado'].values[0]
Monto_Actualizacion = df_ipc_2025.loc[df_ipc_2025['Periodo'] == '12/2025', 'Actualizacion'].values[0]

# Mostrar los resultados con formato
print(f"Monto del Capital al inicio (Enero 2025): ${Capital_Inicial:,.2f}")
print(f"Monto de la Actualización: ${Monto_Actualizacion:,.2f}")
print(f"Monto del Capital Actualizado (Diciembre 2025): ${Monto_Actualizado:,.2f}")

Monto del Capital al inicio (Enero 2025): $1,000,000.00
Monto de la Actualización: $315,492.59
Monto del Capital Actualizado (Diciembre 2025): $1,315,492.59
